In [32]:
import pandas as pd
import matplotlib.pyplot as plt
import re
from pathlib import Path
from config import *

In [33]:
plots_zooms = {
    "lr_0.01_beta1_0.5_beta2_0.2": {
        "zoom_start": 230,
        "zoom_end": 250
    },
}

SUBSTEP_FACTOR = 5 

In [34]:
def extract_params(filename):
    """Extract initial_theta, initial_velocity (optional), lr, beta1, beta2 from filename"""
    # Try to match pattern with initial_velocity (for 2nd order)
    match_with_velocity = re.search(
        r'initial_theta_([-\d.]+?)_initial_velocity_([-\d.]+?)_lr_([\d.]+?)_beta1_([\d.]+?)_beta2_([\d.]+?)(?:\.csv|$)',
        filename
    )
    
    if match_with_velocity:
        initial_theta_str = match_with_velocity.group(1).rstrip('.')
        initial_velocity_str = match_with_velocity.group(2).rstrip('.')
        lr_str = match_with_velocity.group(3).rstrip('.')
        beta1_str = match_with_velocity.group(4).rstrip('.')
        beta2_str = match_with_velocity.group(5).rstrip('.')
        
        return {
            'initial_theta': float(initial_theta_str),
            'initial_velocity': float(initial_velocity_str),
            'lr': float(lr_str),
            'beta1': float(beta1_str),
            'beta2': float(beta2_str)
        }
    
    # Try to match pattern without initial_velocity (for discrete and 1st order)
    match_without_velocity = re.search(
        r'initial_theta_([-\d.]+?)_lr_([\d.]+?)_beta1_([\d.]+?)_beta2_([\d.]+?)(?:\.csv|$)',
        filename
    )
    
    if match_without_velocity:
        initial_theta_str = match_without_velocity.group(1).rstrip('.')
        lr_str = match_without_velocity.group(2).rstrip('.')
        beta1_str = match_without_velocity.group(3).rstrip('.')
        beta2_str = match_without_velocity.group(4).rstrip('.')
        
        return {
            'initial_theta': float(initial_theta_str),
            'initial_velocity': None,  # Will be extracted from m_history[0]
            'lr': float(lr_str),
            'beta1': float(beta1_str),
            'beta2': float(beta2_str)
        }
    
    # Try format: c_X_results_initial_theta_Y_lr_Z_beta1_A_beta2_B.csv (for results/)
    match_results_format = re.search(
        r'results_initial_theta_([-\d.]+?)_lr_([\d.]+?)_beta1_([\d.]+?)_beta2_([\d.]+?)(?:\.csv|$)',
        filename
    )
    
    if match_results_format:
        initial_theta_str = match_results_format.group(1).rstrip('.')
        lr_str = match_results_format.group(2).rstrip('.')
        beta1_str = match_results_format.group(3).rstrip('.')
        beta2_str = match_results_format.group(4).rstrip('.')
        
        return {
            'initial_theta': float(initial_theta_str),
            'initial_velocity': None,  # Will be extracted from m_history[0]
            'lr': float(lr_str),
            'beta1': float(beta1_str),
            'beta2': float(beta2_str)
        }
    
    # Fallback: try old format without initial_theta (for backward compatibility)
    match_old = re.search(r'lr_([\d.]+?)_beta1_([\d.]+?)_beta2_([\d.]+?)(?:\.csv|$)', filename)
    if match_old:
        lr_str = match_old.group(1).rstrip('.')
        beta1_str = match_old.group(2).rstrip('.')
        beta2_str = match_old.group(3).rstrip('.')
        
        return {
            'initial_theta': None,  # Will be extracted from theta_history[0]
            'initial_velocity': None,  # Will be extracted from m_history[0]
            'lr': float(lr_str),
            'beta1': float(beta1_str),
            'beta2': float(beta2_str)
        }
    
    return None

In [35]:
# Define paths to result folders
results_dir = Path("results")
results_1st_order_dir = Path("results_1rst_order")
results_2nd_order_dir = Path("results_2nd_order")

# Read all CSV files from the three folders
discrete_files = list(results_dir.glob("*.csv"))
continuous_1st_files = list(results_1st_order_dir.glob("*.csv"))
continuous_2nd_files = list(results_2nd_order_dir.glob("*.csv"))

In [36]:
# Organize files by parameters (including initial_theta and initial_velocity)
data_dict = {}

# Filter: only process files with beta1=0.5 and beta2=0.2
TARGET_BETA1 = 0.5
TARGET_BETA2 = 0.2
TARGET_INITIAL_VELOCITY = initial_velocity

In [37]:
# Process discrete files
for csv_file in discrete_files:
    params = extract_params(str(csv_file))
    if params is None:
        continue
    
    # Filter by beta values
    if params['beta1'] != TARGET_BETA1 or params['beta2'] != TARGET_BETA2:
        continue
    
    # Create key including initial_theta
    key = (params['lr'], params['beta1'], params['beta2'], params['initial_theta'])
    if key not in data_dict:
        data_dict[key] = {}
    
    df = pd.read_csv(csv_file)
    data_dict[key]['discrete'] = df
    
    # Store initial_theta if not in filename (extract from data)
    if params['initial_theta'] is None:
        params['initial_theta'] = df['theta_history'].iloc[0]
    
    # Store initial_velocity from m_history[0] if not in filename
    if params['initial_velocity'] is None:
        params['initial_velocity'] = df['m_history'].iloc[0]
    
    data_dict[key]['params'] = params

# Process first-order continuous files
for csv_file in continuous_1st_files:
    params = extract_params(str(csv_file))
    if params is None:
        continue
    
    # Filter by beta values
    if params['beta1'] != TARGET_BETA1 or params['beta2'] != TARGET_BETA2:
        continue
    
    key = (params['lr'], params['beta1'], params['beta2'], params['initial_theta'])
    if key not in data_dict:
        data_dict[key] = {}
    
    df = pd.read_csv(csv_file)
    data_dict[key]['continuous_1st'] = df
    
    # Store initial_theta if not in filename
    if params['initial_theta'] is None:
        params['initial_theta'] = df['theta_history'].iloc[0]
    
    # Store initial_velocity from m_history[0] if not in filename
    if params['initial_velocity'] is None:
        params['initial_velocity'] = df['m_history'].iloc[0]
    
    # Update params if not already stored
    if 'params' not in data_dict[key]:
        data_dict[key]['params'] = params

# Process second-order continuous files - store multiple velocities
for csv_file in continuous_2nd_files:
    params = extract_params(str(csv_file))
    if params is None:
        continue
    
    # Filter by beta values
    if params['beta1'] != TARGET_BETA1 or params['beta2'] != TARGET_BETA2:
        continue
    
    # Filter by initial_velocity
    if params['initial_velocity'] is not None:
        # Si el initial_velocity está en el nombre del archivo, filtrar por él
        if abs(params['initial_velocity'] - TARGET_INITIAL_VELOCITY) > 1e-6:  # Usar tolerancia para comparación de floats
            continue
    else:
        # Si no está en el nombre, extraerlo del archivo y comparar
        df_temp = pd.read_csv(csv_file)
        extracted_velocity = df_temp['m_history'].iloc[0]  # O el método que uses para extraer initial_velocity
        if abs(extracted_velocity - TARGET_INITIAL_VELOCITY) > 1e-6:
            continue
        params['initial_velocity'] = extracted_velocity
    
    # For 2nd order, initial_velocity should be in filename
    if params['initial_theta'] is None:
        df_temp = pd.read_csv(csv_file)
        params['initial_theta'] = df_temp['theta_history'].iloc[0]
    
    if params['initial_velocity'] is None:
        df_temp = pd.read_csv(csv_file)
        params['initial_velocity'] = df_temp['m_history'].iloc[0]
    
    key = (params['lr'], params['beta1'], params['beta2'], params['initial_theta'])
    if key not in data_dict:
        data_dict[key] = {}
    
    # Store continuous_2nd as a dictionary keyed by initial_velocity
    if 'continuous_2nd' not in data_dict[key]:
        data_dict[key]['continuous_2nd'] = {}
    
    df = pd.read_csv(csv_file)
    initial_velocity = params['initial_velocity']
    data_dict[key]['continuous_2nd'][initial_velocity] = df

In [38]:
c = c_value

In [41]:
# Define colors for different velocities
velocity_colors = ['purple', 'darkorange', 'magenta', 'darkred', 'darkblue']

# Plot for each parameter combination
for (lr, beta1, beta2, initial_theta), data in data_dict.items():
    # Check that we have at least discrete data
    if 'discrete' not in data:
        print(f"Warning: Missing discrete data for lr={lr}, beta1={beta1}, beta2={beta2}, initial_theta={initial_theta}")
        continue
    
    # Get parameters
    params = data.get('params', {})
    
    # Extract discrete data
    df_disc = data['discrete']
    t_disc = df_disc["t"].to_numpy()
    theta_disc = df_disc["theta_history"].to_numpy()
    m_disc = df_disc["m_history"].to_numpy()
    v_disc = df_disc["v_history"].to_numpy()
    f_disc = (1 - theta_disc)**2 + c * (theta_disc**2 - 1)**2
    
    # Extract first-order continuous data if available
    df_cont_1st = data.get('continuous_1st')
    if df_cont_1st is not None:
        # Map continuous time to discrete time equivalent (divide by SUBSTEP_FACTOR)
        t_cont_1st_raw = df_cont_1st["t"].to_numpy()
        t_cont_1st = t_cont_1st_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        theta_cont_1st = df_cont_1st["theta_history"].to_numpy()
        m_cont_1st = df_cont_1st["m_history"].to_numpy()
        v_cont_1st = df_cont_1st["v_history"].to_numpy()
        f_cont_1st = (1 - theta_cont_1st)**2 + c * (theta_cont_1st**2 - 1)**2
    else:
        t_cont_1st = None
    
    # Extract second-order continuous data (multiple velocities)
    cont_2nd_dict = data.get('continuous_2nd', {})
    
    # Get sorted list of velocities for consistent ordering
    velocities_2nd = sorted(cont_2nd_dict.keys()) if cont_2nd_dict else []
    
    # ========== FIG 1: f(theta_k) vs iteration ==========
    fig = plt.figure(figsize=(12, 7))
    ax = plt.gca()

    me_d = max(1, len(t_disc)//35)
    ax.plot(t_disc, f_disc,
                color='red', linestyle=(0, (6, 3)), linewidth=3.0,
                marker='o', markersize=5, markevery=me_d,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=4, label='Discrete Adam')

    if df_cont_1st is not None:
        me_1 = max(1, len(t_cont_1st)//35)
        ax.plot(t_cont_1st, f_cont_1st,
                    color='blue', linestyle='-', linewidth=2.4, alpha=0.90,
                    marker='s', markersize=5, markevery=me_1,
                    markerfacecolor='none', markeredgewidth=1.2,
                    zorder=3, label='Continuous Adam (1st order)')

    # Plot each 2nd order velocity
    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        f_cont_2nd = (1 - df_cont_2nd["theta_history"].to_numpy())**2 + c * (df_cont_2nd["theta_history"].to_numpy()**2 - 1)**2
        
        me_2 = max(1, len(t_cont_2nd)//35)
        color = velocity_colors[idx % len(velocity_colors)]
        
        ax.plot(t_cont_2nd, f_cont_2nd,
                    color=color, linestyle='-', linewidth=2.6, alpha=0.90,
                    marker='d', markersize=7, markevery=me_2,
                    markerfacecolor='none', markeredgewidth=1.2,
                    zorder=2, label=f'Continuous Adam velocity initial = {initial_velocity:.1f} (2nd order)')

    ax.set_xlabel("iteration", fontsize=12)
    ax.set_ylabel(r"$f(\theta_t)$", fontsize=12)
    ax.set_title(fr"Convergence: $f(\theta_t)$ vs $t$ ($\alpha={lr}$, $\beta_1={beta1}$, $\beta_2={beta2}$, $\theta_0={initial_theta:.2f}$)", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Agregar zoom entre 500 y 600 iteraciones
    zoom_key = f"lr_{lr}_beta1_{beta1}_beta2_{beta2}"
    zoom_config = plots_zooms.get(zoom_key, {})

    # Obtener rangos de zoom (valores por defecto 50-150 si no están definidos)
    zoom_start = zoom_config.get("zoom_start", 50)
    zoom_end = zoom_config.get("zoom_end", 150)
    ax_inset = fig.add_axes([0.55, 0.35, 0.35, 0.35])  # [left, bottom, width, height] en coordenadas de figura (0-1)

    # Filtrar datos en el rango de zoom
    mask_disc = (t_disc >= zoom_start) & (t_disc <= zoom_end)
    ax_inset.plot(t_disc[mask_disc], f_disc[mask_disc],
                    color='red', linestyle=(0, (6, 3)), linewidth=2.5,
                    marker='o', markersize=4, markevery=max(1, sum(mask_disc)//10),
                    markerfacecolor='none', markeredgewidth=1.0,
                    zorder=4)

    if df_cont_1st is not None:
        mask_1st = (t_cont_1st >= zoom_start) & (t_cont_1st <= zoom_end)
        ax_inset.plot(t_cont_1st[mask_1st], f_cont_1st[mask_1st],
                        color='blue', linestyle='-', linewidth=2.0, alpha=0.90,
                        marker='s', markersize=4, markevery=max(1, sum(mask_1st)//10),
                        markerfacecolor='none', markeredgewidth=1.0,
                        zorder=3)

    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR 
        f_cont_2nd = (1 - df_cont_2nd["theta_history"].to_numpy())**2 + c * (df_cont_2nd["theta_history"].to_numpy()**2 - 1)**2
        mask_2nd = (t_cont_2nd >= zoom_start) & (t_cont_2nd <= zoom_end)
        
        color = velocity_colors[idx % len(velocity_colors)]
        
        ax_inset.plot(t_cont_2nd[mask_2nd], f_cont_2nd[mask_2nd],
                        color=color, linestyle='-', linewidth=2.2, alpha=0.90,
                        marker='d', markersize=7, markevery=max(1, sum(mask_2nd)//10),
                        markerfacecolor='none', markeredgewidth=1.0,
                        zorder=2)

    ax_inset.set_xlim(zoom_start, zoom_end)
    ax_inset.grid(True, alpha=0.3)
    ax_inset.set_xlabel("iteration", fontsize=9)
    ax_inset.set_ylabel(r"$f(\theta_t)$", fontsize=9)
    ax_inset.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(f"plots/c_{c_value}_convergence_comparison_lr_{lr}_beta1_{beta1}_beta2_{beta2}_theta0_{initial_theta:.2f}.png", dpi=150)
    plt.close()

    # ========== FIG 2: theta vs iteration ==========
    fig = plt.figure(figsize=(12, 7))
    ax = plt.gca()

    me_d = max(1, len(t_disc)//35)
    ax.plot(t_disc, theta_disc,
            color='red', linestyle=(0, (6, 3)), linewidth=3.0,
            marker='o', markersize=5, markevery=me_d,
            markerfacecolor='none', markeredgewidth=1.2,
            zorder=4, label='Discrete Adam')

    if df_cont_1st is not None:
        me_1 = max(1, len(t_cont_1st)//35)
        ax.plot(t_cont_1st, theta_cont_1st,
                color='blue', linestyle='-', linewidth=2.4, alpha=0.90,
                marker='s', markersize=5, markevery=me_1,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=3, label='Continuous Adam (1st order)')

    # Plot each 2nd order velocity
    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        theta_cont_2nd = df_cont_2nd["theta_history"].to_numpy()
        
        me_2 = max(1, len(t_cont_2nd)//35)
        color = velocity_colors[idx % len(velocity_colors)]
        
        ax.plot(t_cont_2nd, theta_cont_2nd,
                color=color, linestyle='-', linewidth=2.6, alpha=0.90,
                marker='d', markersize=7, markevery=me_2,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=2, label=f'Continuous Adam velocity initial = {initial_velocity:.1f} (2nd order)')

    ax.set_xlabel("iteration", fontsize=12)
    ax.set_ylabel(r"$\theta_t$", fontsize=12)
    ax.set_title(fr"Evolution of $\theta_t$ ($\alpha={lr}$, $\beta_1={beta1}$, $\beta_2={beta2}$, $\theta_0={initial_theta:.2f}$)", fontsize=14)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # Agregar zoom entre 500 y 600 iteraciones
    zoom_key = f"lr_{lr}_beta1_{beta1}_beta2_{beta2}"
    zoom_config = plots_zooms.get(zoom_key, {})

    # Obtener rangos de zoom (valores por defecto 50-150 si no están definidos)
    zoom_start = zoom_config.get("zoom_start", 50)
    zoom_end = zoom_config.get("zoom_end", 150)
    ax_inset = fig.add_axes([0.55, 0.35, 0.35, 0.35])  # [left, bottom, width, height] en coordenadas de figura (0-1)

    # Filtrar datos en el rango de zoom
    mask_disc = (t_disc >= zoom_start) & (t_disc <= zoom_end)
    ax_inset.plot(t_disc[mask_disc], theta_disc[mask_disc],
                    color='red', linestyle=(0, (6, 3)), linewidth=2.5,
                    marker='o', markersize=4, markevery=max(1, sum(mask_disc)//10),
                    markerfacecolor='none', markeredgewidth=1.0,
                    zorder=4)

    if df_cont_1st is not None:
        mask_1st = (t_cont_1st >= zoom_start) & (t_cont_1st <= zoom_end)
        ax_inset.plot(t_cont_1st[mask_1st], theta_cont_1st[mask_1st],
                        color='blue', linestyle='-', linewidth=2.0, alpha=0.90,
                        marker='s', markersize=4, markevery=max(1, sum(mask_1st)//10),
                        markerfacecolor='none', markeredgewidth=1.0,
                        zorder=3)

    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        theta_cont_2nd = df_cont_2nd["theta_history"].to_numpy()
        mask_2nd = (t_cont_2nd >= zoom_start) & (t_cont_2nd <= zoom_end)
        
        color = velocity_colors[idx % len(velocity_colors)]
        
        ax_inset.plot(t_cont_2nd[mask_2nd], theta_cont_2nd[mask_2nd],
                        color=color, linestyle='-', linewidth=2.2, alpha=0.90,
                        marker='d', markersize=7, markevery=max(1, sum(mask_2nd)//10),
                        markerfacecolor='none', markeredgewidth=1.0,
                        zorder=2)

    ax_inset.set_xlim(zoom_start, zoom_end)
    ax_inset.grid(True, alpha=0.3)
    ax_inset.set_xlabel("iteration", fontsize=9)
    ax_inset.set_ylabel(r"$\theta_t$", fontsize=9)
    ax_inset.tick_params(labelsize=8)

    plt.tight_layout()
    plt.savefig(f"plots/c_{c_value}_theta_evolution_comparison_lr_{lr}_beta1_{beta1}_beta2_{beta2}_theta0_{initial_theta:.2f}.png", dpi=150)
    plt.close()
    
    # ========== FIG 3: m_history vs iteration ==========
    plt.figure(figsize=(12, 7))
    me_d = max(1, len(t_disc)//35)

    plt.plot(t_disc, m_disc,
            color='red', linestyle=(0,(6,3)), linewidth=3.0,
            marker='o', markersize=5, markevery=me_d,
            markerfacecolor='none', markeredgewidth=1.2,
            zorder=4, label='Discrete Adam')

    if df_cont_1st is not None:
        me_1 = max(1, len(t_cont_1st)//35)
        plt.plot(t_cont_1st, m_cont_1st,
                color='blue', linestyle='-', linewidth=2.4, alpha=0.90,
                marker='s', markersize=5, markevery=me_1,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=3, label='Continuous Adam (1st order)')

    # Plot each 2nd order velocity
    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        m_cont_2nd = df_cont_2nd["m_history"].to_numpy()
        
        me_2 = max(1, len(t_cont_2nd)//35)
        color = velocity_colors[idx % len(velocity_colors)]
        
        plt.plot(t_cont_2nd, m_cont_2nd,
                color=color, linestyle='-', linewidth=2.6, alpha=0.90,
                marker='d', markersize=7, markevery=me_2,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=2, label=f'Continuous Adam velocity initial = {initial_velocity:.1f} (2nd order)')

    plt.xlabel("iteration", fontsize=12)
    plt.ylabel(r"$m_t$", fontsize=12)
    plt.title(fr"Evolution of the first moment $m_t$ ($\alpha={lr}$, $\beta_1={beta1}$, $\beta_2={beta2}$, $\theta_0={initial_theta:.2f}$)", fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"plots/c_{c_value}_first_moment_comparison_lr_{lr}_beta1_{beta1}_beta2_{beta2}_theta0_{initial_theta:.2f}.png", dpi=150)
    plt.close()

    # ========== FIG 4: v_history vs iteration ==========
    plt.figure(figsize=(12, 7))
    me_d = max(1, len(t_disc)//35)

    plt.plot(t_disc, v_disc,
            color='red', linestyle=(0,(6,3)), linewidth=3.0,
            marker='o', markersize=5, markevery=me_d,
            markerfacecolor='none', markeredgewidth=1.2,
            zorder=4, label='Discrete Adam')

    if df_cont_1st is not None:
        me_1 = max(1, len(t_cont_1st)//35)
        plt.plot(t_cont_1st, v_cont_1st,
                color='blue', linestyle='-', linewidth=2.4, alpha=0.90,
                marker='s', markersize=5, markevery=me_1,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=3, label='Continuous Adam (1st order)')

    # Plot each 2nd order velocity
    for idx, initial_velocity in enumerate(velocities_2nd):
        df_cont_2nd = cont_2nd_dict[initial_velocity]
        t_cont_2nd_raw = df_cont_2nd["t"].to_numpy()
        t_cont_2nd = t_cont_2nd_raw / SUBSTEP_FACTOR  # Convert substeps to discrete iterations
        v_cont_2nd = df_cont_2nd["v_history"].to_numpy()
        
        me_2 = max(1, len(t_cont_2nd)//35)
        color = velocity_colors[idx % len(velocity_colors)]
        
        plt.plot(t_cont_2nd, v_cont_2nd,
                color=color, linestyle='-', linewidth=2.6, alpha=0.90,
                marker='d', markersize=7, markevery=me_2,
                markerfacecolor='none', markeredgewidth=1.2,
                zorder=2, label=f'Continuous Adam velocity initial = {initial_velocity:.1f} (2nd order)')

    plt.xlabel("iteration", fontsize=12)
    plt.ylabel(r"$v_t$", fontsize=12)
    plt.title(fr"Evolution of the second moment $v_t$ ($\alpha={lr}$, $\beta_1={beta1}$, $\beta_2={beta2}$, $\theta_0={initial_theta:.2f}$)", fontsize=14)
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"plots/c_{c_value}_second_moment_comparison_lr_{lr}_beta1_{beta1}_beta2_{beta2}_theta0_{initial_theta:.2f}.png", dpi=150)
    plt.close()
    
    velocities_str = ", ".join([f"v₀={v:.1f}" for v in velocities_2nd])
    print(f"Plots generated for lr={lr}, beta1={beta1}, beta2={beta2}, initial_theta={initial_theta:.2f}, velocities: {velocities_str if velocities_2nd else 'none'}")

/tmp/ipykernel_7032/2524846688.py:128: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_7032/2524846688.py:219: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Plots generated for lr=0.01, beta1=0.5, beta2=0.2, initial_theta=-1.50, velocities: v₀=1.0
